# Grid ISIC certification

Use this notebook to run multi-head grid certification for ISIC grids (per-cell smoothing) and produce a results\_\*.pkl that downstream faithfulness notebooks can consume.


In [1]:
import sys
from pathlib import Path

# Add workspace root to path
root = Path.cwd().resolve()
while root != root.parent:
    if (root / "src").exists() and (root / "certify_grid_isic_server.py").exists():
        break
    root = root.parent

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print(f"Workspace root: {root}")
print(f"sys.path[0]: {sys.path[0]}")

Workspace root: D:\git projects\certified-attribution-medical-imaging
sys.path[0]: D:\git projects\certified-attribution-medical-imaging


In [2]:
import subprocess
import sys

# Fix sympy compatibility issue with torch
print("Fixing sympy compatibility...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "sympy==1.12", "-q"])
print("sympy downgraded to 1.12 for torch compatibility")

Fixing sympy compatibility...
sympy downgraded to 1.12 for torch compatibility


In [3]:
import sys
from pathlib import Path

# Ensure workspace root is in path
root = Path.cwd().resolve()
while root != root.parent:
    if (root / "src").exists() and (root / "certify_grid_isic_server.py").exists():
        break
    root = root.parent

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print(f"Workspace root: {root}")

from datetime import datetime
import pickle

import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

from src.datasets.grid_dataset import GridDataset
from src.certify.smoothing import RandomizedSmoothingAttributor
from certify_grid_isic_server import load_model, build_attr_methods

# Custom collate function to handle meta dicts
def grid_collate_fn(batch):
    """Custom collate that extracts meta dicts separately."""
    images = torch.stack([item["image"] for item in batch])
    cell_classes = torch.stack([item["cell_classes"] for item in batch])
    target_classes = torch.tensor([item["target_class"] for item in batch])
    target_heads = torch.tensor([item["target_head"] for item in batch])
    
    # For batch_size=1, return single meta dict; otherwise return list
    if len(batch) == 1:
        meta = batch[0]["meta"]
    else:
        meta = [item["meta"] for item in batch]
    
    return {
        "image": images,
        "cell_classes": cell_classes,
        "target_class": target_classes,
        "target_head": target_heads,
        "meta": meta,
    }

# Configuration (all paths relative to workspace root)
cfg = {
    "grid_pt": root / "data/raw/grid/isic/val/grid.pt",
    "checkpoint": root / "notebooks/output/checkpoints/isic/resnet18/final_model.pt",
    "num_classes": 8,
    "device": "auto",  # "auto" -> cuda if available else cpu
    "sigma": 0.15,
    "num_samples": 10,
    "tau": 0.75,
    "alpha": 0.001,
    "batch_size": 1,
    "k_percents": [50, 25, 5],
    "save_dir": root / "notebooks/output/bulk_certifcation/grid/isic/resnet18",
    "save_noisy_samples": False,
    "max_noisy_samples": 3,
    "max_items": 2,  # LIMIT TO 2 SAMPLES FOR QUICK TEST - increase to None for full dataset
}

cfg["device"] = cfg["device"] if cfg["device"] != "auto" else ("cuda" if torch.cuda.is_available() else "cpu")
print(f"Grid PT exists: {cfg['grid_pt'].exists()}")
print(f"Checkpoint exists: {cfg['checkpoint'].exists()}")
print(f"Max items to process: {cfg['max_items']}")
print(cfg)

Workspace root: D:\git projects\certified-attribution-medical-imaging
Grid PT exists: True
Checkpoint exists: True
Max items to process: 2
{'grid_pt': WindowsPath('D:/git projects/certified-attribution-medical-imaging/data/raw/grid/isic/val/grid.pt'), 'checkpoint': WindowsPath('D:/git projects/certified-attribution-medical-imaging/notebooks/output/checkpoints/isic/resnet18/final_model.pt'), 'num_classes': 8, 'device': 'cpu', 'sigma': 0.15, 'num_samples': 10, 'tau': 0.75, 'alpha': 0.001, 'batch_size': 1, 'k_percents': [50, 25, 5], 'save_dir': WindowsPath('D:/git projects/certified-attribution-medical-imaging/notebooks/output/bulk_certifcation/grid/isic/resnet18'), 'save_noisy_samples': False, 'max_noisy_samples': 3, 'max_items': 2}


In [4]:
grid_ds = GridDataset(cfg["grid_pt"])

print(f"Dataset size: {len(grid_ds)}")
print(f"Scale: {grid_ds.scale} (heads={grid_ds.scale * grid_ds.scale}) target_cell={grid_ds.target_cell}")
sample = grid_ds[0]
print(f"Sample keys: {list(sample.keys())}")
print(f"Image shape: {tuple(sample['image'].shape)}")
print(f"Target class: {int(sample['target_class'])}, target head: {int(sample['target_head'])}")

Dataset size: 32
Scale: 2 (heads=4) target_cell=0
Sample keys: ['image', 'cell_classes', 'target_class', 'target_head', 'meta']
Image shape: (3, 448, 448)
Target class: 1, target head: 0


In [ ]:
def run_grid_certification(cfg):
    ds = GridDataset(cfg["grid_pt"])
    num_heads = ds.scale * ds.scale
    loader = DataLoader(ds, batch_size=cfg["batch_size"], shuffle=False, collate_fn=grid_collate_fn)
    
    checkpoint = str(cfg["checkpoint"]) if cfg.get("checkpoint") else None
    model = load_model(
        num_heads=num_heads,
        num_classes=cfg["num_classes"],
        device=cfg["device"],
        checkpoint=checkpoint,
    )
    
    results = {}
    start = datetime.now()
    for batch_idx, sample in enumerate(tqdm(loader, desc="Certifying grids", total=len(loader))):
        if cfg["max_items"] is not None and batch_idx >= cfg["max_items"]:
            break
        
        image = sample["image"].to(cfg["device"])
        target_class = int(sample["target_class"].item())
        head_id = int(sample["target_head"].item())
        meta = sample["meta"] if isinstance(sample["meta"], dict) else sample["meta"][0]
        scale = meta.get("scale", ds.scale)
        
        # Get image dimensions for resizing attributions
        _, _, img_h, img_w = image.shape
        
        methods = build_attr_methods(model, cfg["device"], head_id)
        smoother = RandomizedSmoothingAttributor(model, None, device=cfg["device"])
        
        def make_attr_wrapper(attr_obj):
            def _fn(img, target_class_override=None):
                tc = target_class if target_class_override is None else int(target_class_override)
                attr = attr_obj.attribute(img, target_class=tc)
                # Ensure attribution matches image size
                if isinstance(attr, torch.Tensor):
                    attr = attr.detach().cpu().numpy()
                attr = np.array(attr)
                # Reduce to 2D if needed
                if attr.ndim == 4:
                    attr = attr.squeeze(0)
                if attr.ndim == 3:
                    attr = attr.mean(axis=0)
                # Resize to match image size
                if attr.shape != (img_h, img_w):
                    from scipy.ndimage import zoom
                    zoom_factors = (img_h / attr.shape[0], img_w / attr.shape[1])
                    attr = zoom(attr, zoom_factors, order=1)
                # Normalize to [0, 1]
                attr_min, attr_max = attr.min(), attr.max()
                if attr_max > attr_min:
                    attr = (attr - attr_min) / (attr_max - attr_min)
                else:
                    attr = np.zeros_like(attr)
                return torch.from_numpy(attr).to(cfg["device"])
            return _fn
        
        for mname, attr_obj in methods.items():
            attr_wrapper = make_attr_wrapper(attr_obj)
            for k in cfg["k_percents"]:
                smoother.attribution_func = attr_wrapper
                res = smoother.certify(
                    image,
                    k_percent=k,
                    target_class=target_class,
                    sigma=cfg["sigma"],
                    num_samples=cfg["num_samples"],
                    tau=cfg["tau"],
                    batch_size=cfg["batch_size"],
                    alpha=cfg["alpha"],
                    save_noisy_samples=cfg["save_noisy_samples"],
                    max_noisy_samples=cfg["max_noisy_samples"],
                )
                results.setdefault(mname, {}).setdefault(k, []).append({
                    "image_idx": batch_idx,
                    "label": target_class,
                    "head_id": head_id,
                    "scale": scale,
                    "target_cell": ds.target_cell,
                    "results": res,
                })
    
    cfg["save_dir"].mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    pkl_path = cfg["save_dir"] / f"results_{ts}.pkl"
    with open(pkl_path, "wb") as f:
        pickle.dump({"resnet18": results}, f)
    elapsed = datetime.now() - start
    print(f"Saved certification results to {pkl_path} (elapsed {elapsed})")
    return pkl_path, results


import numpy as np
pkl_path, cert_results = run_grid_certification(cfg)

d:\git projects\certified-attribution-medical-imaging\venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\git projects\certified-attribution-medical-imaging\venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Certifying grids:   0%|          | 0/32 [00:00<?, ?it/s]d:\git projects\certified-attribution-medical-imaging\venv\Lib\site-packages\torch\nn\modules\module.py:1866: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the document

[DEBUG] GradCAM fix active: hooks clone tensors

[Randomized Smoothing] σ=0.15, τ=0.75, n=10, K=50%


[DEBUG] GradCAM fix active: hooks clone tensors


[DEBUG] GradCAM fix active: hooks clone tensors


[DEBUG] GradCAM fix active: hooks clone tensors


...

[Results] Certified: 36.5% | Abstained: 63.5% | Radius: 0.1012

[Randomized Smoothing] σ=0.15, τ=0.75, n=10, K=25%


Smoothing samples: 100%|██████████| 10/10 [10:04<00:00, 60.43s/it]


[Results] Certified: 55.5% | Abstained: 44.5% | Radius: 0.1012

[Randomized Smoothing] σ=0.15, τ=0.75, n=10, K=5%


Smoothing samples: 100%|██████████| 10/10 [09:46<00:00, 58.63s/it]


[Results] Certified: 87.2% | Abstained: 12.8% | Radius: 0.1012

[Randomized Smoothing] σ=0.15, τ=0.75, n=10, K=50%


In [ ]:
pkl_files = sorted(cfg["save_dir"].glob("results_*.pkl"))
if pkl_files:
    latest = pkl_files[-1]
    print(f"Latest certification file: {latest}")
else:
    print("No certification files yet; run the cell above to generate one.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import ListedColormap

# Load the latest results
pkl_files = sorted(cfg["save_dir"].glob("results_*.pkl"))
if pkl_files:
    latest_pkl = pkl_files[-1]
    with open(latest_pkl, "rb") as f:
        data = pickle.load(f)
    cert_results = data["resnet18"]
    print(f"Loaded results from: {latest_pkl}")
else:
    print("No results found!")
    cert_results = None

if cert_results:
    # Get first method and k_percent available
    methods = list(cert_results.keys())
    print(f"Available methods: {methods}")
    
    for method_name in methods[:1]:  # Visualize first method
        k_values = list(cert_results[method_name].keys())
        print(f"k_percents for {method_name}: {k_values}")
        
        for k in k_values[:1]:  # Visualize first k_percent
            results_list = cert_results[method_name][k]
            
            # Visualize each sample
            for sample_idx, sample_result in enumerate(results_list[:2]):
                res = sample_result["results"]
                label = sample_result["label"]
                head_id = sample_result["head_id"]
                
                # Extract certification maps and heatmaps
                certified_map = res["certified_map"]
                p_1 = res["p_1"]
                heatmap_clean = res["heatmap_clean"]
                ss_map = res["ss_map"]
                certified_radius = res["certified_radius"]
                pct_certified = res["pct_certified"]
                pct_abstained = res["pct_abstained"]
                
                # Create visualization
                fig, axes = plt.subplots(2, 3, figsize=(15, 10))
                fig.suptitle(f"Grid Certification Results - Sample {sample_idx} (Method: {method_name}, K={k}%)\n"
                             f"Label: {label}, Head: {head_id}, Radius: {certified_radius:.4f}", 
                             fontsize=14, fontweight='bold')
                
                # 1. Clean heatmap
                im0 = axes[0, 0].imshow(heatmap_clean, cmap='hot')
                axes[0, 0].set_title("Clean Attribution")
                axes[0, 0].axis('off')
                plt.colorbar(im0, ax=axes[0, 0])
                
                # 2. Smoothed sparsified map
                im1 = axes[0, 1].imshow(ss_map, cmap='viridis')
                axes[0, 1].set_title("Smoothed Sparsified (SS)")
                axes[0, 1].axis('off')
                plt.colorbar(im1, ax=axes[0, 1])
                
                # 3. Probability of class 1
                im2 = axes[0, 2].imshow(p_1, cmap='coolwarm', vmin=0, vmax=1)
                axes[0, 2].set_title("P(class=1)")
                axes[0, 2].axis('off')
                plt.colorbar(im2, ax=axes[0, 2])
                
                # 4. Certified map (with custom colormap)
                cmap = ListedColormap(['gray', 'blue', 'red'])
                im3 = axes[1, 0].imshow(certified_map, cmap=cmap, vmin=-1, vmax=1)
                axes[1, 0].set_title(f"Certified Map\n(Gray=abstain, Red=cert_1, Blue=cert_0)")
                axes[1, 0].axis('off')
                
                # 5. Certification statistics
                axes[1, 1].axis('off')
                stats_text = (
                    f"Certification Statistics\n"
                    f"{'='*30}\n"
                    f"Certified: {pct_certified:.1f}%\n"
                    f"Abstained: {pct_abstained:.1f}%\n"
                    f"Certified Radius: {certified_radius:.4f}\n"
                    f"{'='*30}\n"
                    f"Certified-1: {res['pct_certified_1']:.1f}%\n"
                    f"Certified-0: {res['pct_certified_0']:.1f}%\n"
                    f"σ={res['stats']['sigma']}\n"
                    f"τ={res['stats']['tau']}\n"
                    f"α={res['stats']['alpha']}"
                )
                axes[1, 1].text(0.1, 0.5, stats_text, fontsize=11, family='monospace',
                               verticalalignment='center',
                               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
                
                # 6. Certified regions overlay on clean heatmap
                im5 = axes[1, 2].imshow(heatmap_clean, cmap='gray', alpha=0.6)
                # Overlay certified regions
                certified_1_mask = (certified_map == 1).astype(float)
                certified_0_mask = (certified_map == 0).astype(float)
                axes[1, 2].imshow(certified_1_mask, cmap=ListedColormap(['none', 'red']), alpha=0.4)
                axes[1, 2].imshow(certified_0_mask, cmap=ListedColormap(['none', 'blue']), alpha=0.4)
                axes[1, 2].set_title("Certified Regions Overlay")
                axes[1, 2].axis('off')
                
                plt.tight_layout()
                plt.show()
                
                print(f"\nSample {sample_idx}: Certified={pct_certified:.1f}%, "
                      f"Abstained={pct_abstained:.1f}%, Radius={certified_radius:.4f}")